In [1]:
import cv2
import os
import numpy as np
from mtcnn import MTCNN
from keras_facenet import FaceNet


c:\Users\jabba\OneDrive\Desktop\facial_recognition_project\venv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
detector = MTCNN()
embedder = FaceNet()

print("MTCNN and FaceNet loaded")



MTCNN and FaceNet loaded


In [3]:
dataset_path = "dataset"

known_embeddings = []
known_names = []

processed = 0
skipped = 0

print("📁 Persons found in dataset:")

for person_name in os.listdir(dataset_path):
    person_dir = os.path.join(dataset_path, person_name)

    if not os.path.isdir(person_dir):
        continue

    print(" -", person_name)

    for img_name in os.listdir(person_dir):
        img_path = os.path.join(person_dir, img_name)

        img = cv2.imread(img_path)
        if img is None:
            skipped += 1
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        h, w, _ = img.shape
        if h > 1000 or w > 1000:
            img = cv2.resize(img, (800, 800))

        try:
            faces = detector.detect_faces(img)
        except:
            skipped += 1
            continue

        if len(faces) == 0:
            skipped += 1
            continue

        x, y, w, h = faces[0]['box']
        x, y = max(0, x), max(0, y)
        face = img[y:y+h, x:x+w]

        if face.size == 0:
            skipped += 1
            continue

        face = cv2.resize(face, (160, 160))
        face = face.astype("float32")

        # ✅ Correct FaceNet call (NO verbose)
        embedding = embedder.embeddings([face])[0]

        known_embeddings.append(embedding)
        known_names.append(person_name)
        processed += 1

print("\n✅ Embedding extraction completed")
print("✔ Total faces processed:", processed)
print("⚠ Images skipped:", skipped)


📁 Persons found in dataset:
 - Adeeb
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step
 - Harshith
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
 - Jabbar
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
1/1 ━━━━━

In [4]:
os.makedirs("embeddings", exist_ok=True)

np.save("embeddings/face_embeddings.npy", known_embeddings)
np.save("embeddings/face_names.npy", known_names)

print("Embeddings saved successfully")


Embeddings saved successfully


In [5]:
known_embeddings = np.load("embeddings/face_embeddings.npy")
known_names = np.load("embeddings/face_names.npy")

print("Embeddings loaded")
print("Total known faces:", len(known_names))


Embeddings loaded
Total known faces: 58


In [6]:
from sklearn.metrics.pairwise import cosine_similarity

THRESHOLD = 0.75

cap = cv2.VideoCapture(0)

print("Starting live face recognition...")
print("Press 'q' to quit")

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    # Convert BGR → RGB (IMPORTANT)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Resize very large frames to avoid MTCNN crash
    h, w, _ = rgb_frame.shape
    if h > 1000 or w > 1000:
        rgb_frame = cv2.resize(rgb_frame, (800, 800))
        frame = cv2.resize(frame, (800, 800))

    try:
        faces = detector.detect_faces(rgb_frame)
    except:
        # Skip bad frame safely
        cv2.imshow("Facial Recognition System", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        continue

    for face_data in faces:
        x, y, w, h = face_data['box']
        x, y = max(0, x), max(0, y)

        face = rgb_frame[y:y+h, x:x+w]
        if face.size == 0:
            continue

        face = cv2.resize(face, (160, 160))
        face = face.astype("float32")

        embedding = embedder.embeddings([face])[0]

        similarities = cosine_similarity([embedding], known_embeddings)[0]
        best_match = np.argmax(similarities)
        best_score = similarities[best_match]

        if best_score > THRESHOLD:
            name = known_names[best_match]
        else:
            name = "Unknown"

        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(frame, name, (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)

    cv2.imshow("Facial Recognition System", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Starting live face recognition...
Press 'q' to quit
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 180ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 157ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step
1/1 ━━━━━━━━